In [0]:
from pyspark.sql.functions import current_timestamp

file_path = "/Volumes/bronze_dev/global_mart_retail/raw_data/Superstore.csv"

In [0]:
from pyspark.sql.functions import col

# reading csv with inferSchema and header
df_raw = spark.read.format('csv')\
                    .option('header', 'true')\
                    .option('inferSchema','true')\
                    .load(file_path)

# adding metadata columns for governance and lineage
df_bronze = df_raw.withColumn('Ingest Timestamp',current_timestamp())\
                    .withColumn('Source File', col('_metadata.file_path'))

In [0]:
df_bronze.show(5)

In [0]:
# standardise the naming of columns
for col in df_bronze.columns:
    new_col = col.title().replace(' ', '').replace('-','')
    df_bronze = df_bronze.withColumnRenamed(col, new_col)

In [0]:
# saving as delta table
df_bronze.write.mode("overwrite").saveAsTable("bronze_dev.global_mart_retail.orders_bronze")

print("Bronze Table Created Successfully!")

In [0]:
%sql
select distinct CustomerId, CustomerName, Segment, Region, Country, State, City, PostalCode, OrderDate
from bronze_dev.global_mart_retail.orders_bronze
where CustomerId in ('CG-12520', 'DV-13045', 'SO-20335')
order by CustomerId, OrderDate

In [0]:
%sql
select distinct ProductId, ProductName, Country, OrderDate
from bronze_dev.global_mart_retail.orders_bronze
where ProductId in ('FUR-CH-10001146', 'OFF-BI-10004654', 'TEC-AC-10003832', 'OFF-ST-10001228')
order by ProductId, Country, OrderDate